# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides guidance for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
This dataset is accessible via a Croissant schema URL and provides record sets covering clinical features, hormone levels & imaging, and genetic variants.

In [ ]:
# Ensure `mlcroissant` library is installed. Remove the bang (`!`) for environments where that is not allowed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate the record sets and their fields using their Croissant `@id`. This ensures reproducibility across schema evolutions.

In [ ]:
# Get all available record sets using their @id values
record_sets = dataset.metadata.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

print("Available Record Sets and their Fields:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']} -- Name: {rs['name'] if 'name' in rs else 'N/A'}")
    # List fields
    if 'fields' in rs:
        for f in rs['fields']:
            print(f"     Field @id: {f['@id']} -- Name: {f['name'] if 'name' in f else 'N/A'}")
    else:
        print("     No fields listed.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities are referenced by their `@id`s from the overview above.

Below, data is extracted for each record set. Use the DataFrame for downstream analysis.

In [ ]:
# Identify all record set @ids
record_sets_ids = record_set_ids  # From above, list of @id
dataframes = {}

# Load data for each record set by @id
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Columns in record set {record_set_id}: {df.columns.tolist()}")
    dataframes[record_set_id] = df

# Preview one record set
if record_sets_ids:
    example_record_set_id = record_sets_ids[0]
    print(f"Sample data from {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select the first available numeric field for demonstration, filter for values above a threshold, and normalize.

In [ ]:
# Choose a numeric field for EDA from the first record set
eda_record_set_id = example_record_set_id
df = dataframes[eda_record_set_id]

# Try to select a plausible numeric field
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    # Optionally group by another field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field (if present), or relationships between fields in the dataset.

In [ ]:
# If a numeric field exists, show histogram
if numeric_field:
    plt.figure(figsize=(6, 4))
    df[numeric_field].hist(bins=10)
    plt.title(f'Distribution of {numeric_field} in Record Set {eda_record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If a group_field is also selected, show boxplot
    if group_field:
        plt.figure(figsize=(6, 4))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 dataset, referencing all entities by their `@id`s using `mlcroissant`. We examined available record sets, processed fields, and visualized key distributions. For more advanced analysis, access each record set and field via their `@id` to ensure reproducibility and consistency.

Key findings should focus on the genotype–phenotype heterogeneity, hormone levels, and clinical traits per the dataset scope. Due to its small scale and cohort specificity, it is recommended only for academic explorations and not for predictive ML model training.
